In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # suppress TF info logs

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error


In [23]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 60)
print("  CO2 Hydrogenation — NN Surrogate Model v2")
print("  TensorFlow", tf.__version__)
print("=" * 60)

  CO2 Hydrogenation — NN Surrogate Model v2
  TensorFlow 2.21.0


### SECTION 1: LOAD & FEATURE ENGINEERING

In [24]:
df=pd.read_csv('co2_hydrogenation_dataset_3k.csv')
df = df[df['thermo_ok'] == True].reset_index(drop=True)
print(f"\nDataset loaded: {len(df)} rows")

# ── Fix 1: Engineered features ────────────────────────────────────────────────.
# Raw Keq spans 0.00003→0.018 (600x range) — NN can't learn this directly.
# log(Keq) linearizes the relationship with X_CO2
# T_K² and T_K×P capture nonlinear and interaction effect

df['log_Keq']      = np.log(df['Keq'])
df['log_T_K']      = np.log(df['T_K'])
df['T_K_sq']       = df['T_K'] ** 2
df['T_P_interact'] = df['T_K'] * df['P_bar']

INPUT_COLS = [
    'T_K', 'P_bar', 'H2_CO2_ratio', 'Keq',   # original 4
    'log_Keq', 'log_T_K', 'T_K_sq', 'T_P_interact'  # 4 engineered
]
OUTPUT_COLS = ['X_CO2_pct', 'alpha_outlet', 'S_C2_C4_pct']

print(f"Inputs  ({len(INPUT_COLS)}): {INPUT_COLS}")
print(f"Outputs ({len(OUTPUT_COLS)}): {OUTPUT_COLS}")

X_raw = df[INPUT_COLS].values.astype(np.float32)
y_raw = df[OUTPUT_COLS].values.astype(np.float32)

y_log = y_raw.copy()
y_log[:, 0] = np.log1p(y_raw[:, 0])   # log(1 + X_CO2)



Dataset loaded: 3000 rows
Inputs  (8): ['T_K', 'P_bar', 'H2_CO2_ratio', 'Keq', 'log_Keq', 'log_T_K', 'T_K_sq', 'T_P_interact']
Outputs (3): ['X_CO2_pct', 'alpha_outlet', 'S_C2_C4_pct']


In [31]:
df.head()

,T_K,T_C,P_bar,H2_CO2_ratio,Keq,Q_over_Keq,thermo_ok,r_RWGS_inlet,r_FT_inlet,X_CO2_pct,...,S_C2_C4_pct,Y_olefin_pct,P_CO2_out_bar,P_H2_out_bar,P_CO_out_bar,P_H2O_out_bar,log_Keq,log_T_K,T_K_sq,T_P_interact
0,523.15,250.0,5.0,1.00,0.000032,0.9943,True,0.032009,0.0,0.8127,...,64.4644,0.5239,2.487067,2.457274,0.005484,0.035278,-10.349775,6.259868,273685.9225,2615.75
1,523.15,250.0,5.0,1.44,0.000032,0.9934,True,0.028471,0.0,0.9706,...,64.4678,0.6257,2.031460,2.914248,0.005447,0.034379,-10.349775,6.259868,273685.9225,2615.75
2,523.15,250.0,5.0,1.89,0.000032,0.9914,True,0.025341,0.0,1.0878,...,64.4698,0.7013,1.716551,3.232224,0.005415,0.032346,-10.349775,6.259868,273685.9225,2615.75
3,523.15,250.0,5.0,2.33,0.000032,0.9900,True,0.022724,0.0,1.1808,...,64.4711,0.7613,1.485953,3.466162,0.005389,0.030127,-10.349775,6.259868,273685.9225,2615.75
4,523.15,250.0,5.0,2.78,0.000032,0.9889,True,0.020550,0.0,1.2582,...,64.4721,0.8112,1.309837,3.645457,0.005366,0.028016,-10.349775,6.259868,273685.9225,2615.75


In [38]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))  # Explicitly create a figure window
plt.plot(df['Keq'], df['T_C'])
plt.draw()                  # Force a redraw of the elements
plt.show()

### SECTION 2: TRAIN / VAL / TEST SPLIT

In [25]:
X_tr, X_tmp, y_tr, y_tmp, y_tr_orig, y_tmp_orig = train_test_split(
    X_raw, y_log, y_raw, test_size=0.30, random_state=SEED)

X_vl, X_te, y_vl, y_te, y_vl_orig, y_te_orig = train_test_split(
    X_tmp, y_tmp, y_tmp_orig, test_size=0.50, random_state=SEED)

print(f"\nSplit → Train:{len(X_tr)} | Val:{len(X_vl)} | Test:{len(X_te)}")



Split → Train:2100 | Val:450 | Test:450


### SECTION 3: NORMALIZATION (fit on train only)

In [26]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_tr_n = scaler_X.fit_transform(X_tr).astype(np.float32)
X_vl_n = scaler_X.transform(X_vl).astype(np.float32)
X_te_n = scaler_X.transform(X_te).astype(np.float32)

y_tr_n = scaler_y.fit_transform(y_tr).astype(np.float32)
y_vl_n = scaler_y.transform(y_vl).astype(np.float32)

y_min_v = scaler_y.data_min_.astype(np.float32)
y_max_v = scaler_y.data_max_.astype(np.float32)

print("\nInput feature ranges:")
for i, col in enumerate(INPUT_COLS):
    print(f"  {col:20s}: [{X_raw[:,i].min():.4f}, {X_raw[:,i].max():.4f}]")


Input feature ranges:
  T_K                 : [523.1500, 773.1500]
  P_bar               : [5.0000, 30.0000]
  H2_CO2_ratio        : [1.0000, 5.0000]
  Keq                 : [0.0000, 0.0238]
  log_Keq             : [-10.3498, -3.7390]
  log_T_K             : [6.2599, 6.6505]
  T_K_sq              : [273685.9375, 597760.9375]
  T_P_interact        : [2615.7500, 23194.5000]


### SECTION 4: MODEL ARCHITECTURE

In [27]:
def build_model(n_in=8):
    """
    Shared backbone  : 8 → 128 → 256 → 128 → 64
    X_CO2 head       : 64 → 32 → 1
    alpha head       : 64 → 16 → 1
    S_C2_C4 head     : 64 → 16 → 1
    """
    inp = keras.Input(shape=(n_in,), name='reactor_inputs')

    # ── Shared Backbone ───────────────────────────────────────────────────────

    # Block 1: 8 → 128
    x = layers.Dense(128, kernel_initializer='he_uniform',
                      kernel_regularizer=regularizers.l2(1e-4),
                      name='hidden_1')(inp)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Activation('relu', name='relu_1')(x)
    x = layers.Dropout(0.10, name='drop_1')(x)

    # Block 2: 128 → 256  (widest — learns complex T×P×ratio interactions)
    x = layers.Dense(256, kernel_initializer='he_uniform',
                      kernel_regularizer=regularizers.l2(1e-4),
                      name='hidden_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    x = layers.Activation('relu', name='relu_2')(x)
    x = layers.Dropout(0.10, name='drop_2')(x)

    # Block 3: 256 → 128
    x = layers.Dense(128, kernel_initializer='he_uniform',
                      kernel_regularizer=regularizers.l2(1e-4),
                      name='hidden_3')(x)
    x = layers.BatchNormalization(name='bn_3')(x)
    x = layers.Activation('relu', name='relu_3')(x)
    x = layers.Dropout(0.10, name='drop_3')(x)

    # Block 4: 128 → 64  (compression before heads)
    x = layers.Dense(64, kernel_initializer='he_uniform',
                      kernel_regularizer=regularizers.l2(1e-4),
                      name='hidden_4')(x)
    x = layers.BatchNormalization(name='bn_4')(x)
    x = layers.Activation('relu', name='relu_4')(x)
    x = layers.Dropout(0.05, name='drop_4')(x)

    # ── Separate Output Heads ─────────────────────────────────────────────────

    # X_CO2 head: larger (32 neurons) because it's the hardest to learn
    x_co2_h  = layers.Dense(32, activation='relu',
                             name='xco2_head')(x)
    out_xco2 = layers.Dense(1, name='out_xco2')(x_co2_h)

    # alpha head
    alpha_h   = layers.Dense(16, activation='relu',
                              name='alpha_head')(x)
    out_alpha = layers.Dense(1, name='out_alpha')(alpha_h)

    # S_C2_C4 head
    sc24_h   = layers.Dense(16, activation='relu',
                             name='sc24_head')(x)
    out_sc24 = layers.Dense(1, name='out_sc24')(sc24_h)

    # Concatenate all outputs
    outputs = layers.Concatenate(name='all_outputs')(
        [out_xco2, out_alpha, out_sc24])

    model = keras.Model(inputs=inp, outputs=outputs,
                        name='CO2_Surrogate_v2')
    return model


model = build_model(n_in=len(INPUT_COLS))
model.summary()
print(f"\nTotal params: {model.count_params():,}")

Model: "CO2_Surrogate_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ reactor_inputs (InputLayer)   │ (None, 8)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ hidden_1 (Dense)              │ (None, 128)               │           1,152 │ reactor_inputs[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_1 (BatchNormalization)     │ (None, 128)               │             512 │ hidden_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ relu_1 (Activation)           │ (None, 128)               │               0 │ bn_1[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ drop_1 (Dropout)              │ (None, 128)               │               0 │ relu_1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ hidden_2 (Dense)              │ (None, 256)               │          33,024 │ drop_1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_2 (BatchNormalization)     │ (None, 256)               │           1,024 │ hidden_2[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ relu_2 (Activation)           │ (None, 256)               │               0 │ bn_2[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ drop_2 (Dropout)              │ (None, 256)               │               0 │ relu_2[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ hidden_3 (Dense)              │ (None, 128)               │          32,896 │ drop_2[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_3 (BatchNormalization)     │ (None, 128)               │             512 │ hidden_3[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ relu_3 (Activation)           │ (None, 128)               │               0 │ bn_3[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ drop_3 (Dropout)              │ (None, 128)               │               0 │ relu_3[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ hidden_4 (Dense)              │ (None, 64)                │           8,256 │ drop_3[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_4 (BatchNormalization)     │ (None, 64)                │             256 │ hidden_4[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ relu_4 (Activation)           │ (None, 64)                │               0 │ bn_4[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ drop_4 (Dropout)              │ (None, 64)                │               0 │ relu_4[0][0]               │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 81,859 (319.76 KB)

 Trainable params: 80,707 (315.26 KB)

 Non-trainable params: 1,152 (4.50 KB)


Total params: 81,859


### SECTION 5: CUSTOM WEIGHTED LOSS

In [28]:
OUTPUT_WEIGHTS = tf.constant([3.0, 1.0, 1.0], dtype=tf.float32)

@tf.function
def weighted_mse(y_true, y_pred):
    """MSE with X_CO2 weighted 3x more than alpha and S_C2_C4."""
    sq_err   = tf.square(y_true - y_pred)     # (batch, 3)
    weighted = sq_err * OUTPUT_WEIGHTS         # upweight X_CO2 column
    return tf.reduce_mean(weighted)


### SECTION 6: CUSTOM TRAINING LOOP

In [29]:
optimizer = keras.optimizers.Adam(learning_rate=1e-3)

X_tr_tf = tf.constant(X_tr_n)
y_tr_tf = tf.constant(y_tr_n)
X_vl_tf = tf.constant(X_vl_n)
y_vl_tf = tf.constant(y_vl_n)

EPOCHS     = 600
BATCH_SIZE = 32
PATIENCE   = 60
LR_PAT     = 25
LR_FACTOR  = 0.5
LR_MIN     = 1e-6

best_val_loss = np.inf
best_weights  = None
wait    = 0
lr_wait = 0
cur_lr  = 1e-3

n_train = X_tr_tf.shape[0]
n_batch = int(np.ceil(n_train / BATCH_SIZE))

history = {'train_loss': [], 'val_loss': []}

print(f"\nTraining: {EPOCHS} epochs | batch={BATCH_SIZE} | patience={PATIENCE}")
print("-" * 60)


@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        y_pred = model(x_batch, training=True)
        loss   = weighted_mse(y_batch, y_pred)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss


@tf.function
def val_step(x_batch, y_batch):
    y_pred = model(x_batch, training=False)
    return weighted_mse(y_batch, y_pred)


for epoch in range(1, EPOCHS + 1):

    # Shuffle training data
    idx    = tf.random.shuffle(tf.range(n_train))
    X_shuf = tf.gather(X_tr_tf, idx)
    y_shuf = tf.gather(y_tr_tf, idx)

    # Mini-batch training
    batch_losses = []
    for b in range(n_batch):
        xb = X_shuf[b * BATCH_SIZE : (b+1) * BATCH_SIZE]
        yb = y_shuf[b * BATCH_SIZE : (b+1) * BATCH_SIZE]
        batch_losses.append(float(train_step(xb, yb)))

    train_loss = np.mean(batch_losses)
    val_loss   = float(val_step(X_vl_tf, y_vl_tf))

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    # Early stopping + LR scheduler
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_weights  = model.get_weights()
        wait    = 0
        lr_wait = 0
    else:
        wait    += 1
        lr_wait += 1

    if lr_wait >= LR_PAT:
        cur_lr = max(cur_lr * LR_FACTOR, LR_MIN)
        optimizer.learning_rate.assign(cur_lr)
        lr_wait = 0

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}/{EPOCHS} | "
              f"train={train_loss:.5f} | val={val_loss:.5f} | "
              f"lr={cur_lr:.2e}")

    if wait >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} "
              f"(best val_loss={best_val_loss:.5f})")
        break

# Restore best weights
model.set_weights(best_weights)
print(f"\nBest weights restored. Best val_loss = {best_val_loss:.5f}")


Training: 600 epochs | batch=32 | patience=60
------------------------------------------------------------
Epoch    1/600 | train=0.19042 | val=0.23446 | lr=1.00e-03
Epoch   50/600 | train=0.00528 | val=0.00069 | lr=1.00e-03
Epoch  100/600 | train=0.00292 | val=0.00050 | lr=1.00e-03
Epoch  150/600 | train=0.00205 | val=0.00071 | lr=5.00e-04
Epoch  200/600 | train=0.00169 | val=0.00032 | lr=2.50e-04

Early stopping at epoch 240 (best val_loss=0.00023)

Best weights restored. Best val_loss = 0.00023


### SECTION 7: EVALUATION ON TEST SET

In [30]:
print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

y_pred_n = model.predict(X_te_n, verbose=0)

# Denormalize from scaled log space back to physical space
y_pred_log  = y_pred_n * (y_max_v - y_min_v) + y_min_v
y_pred_phys = y_pred_log.copy()
y_pred_phys[:, 0] = np.expm1(y_pred_log[:, 0])   # inverse of log1p

y_true_phys = y_te_orig

output_names = ['X_CO2 (%)', 'alpha', 'S_C2_C4 (%)']
for i, name in enumerate(output_names):
    yt   = y_true_phys[:, i]
    yp   = y_pred_phys[:, i]
    r2   = r2_score(yt, yp)
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    print(f"\n  {name}:")
    print(f"    R²   = {r2:.4f}")
    print(f"    MAE  = {mae:.4f}")
    print(f"    RMSE = {rmse:.4f}")


TEST SET EVALUATION

  X_CO2 (%):
    R²   = 0.9961
    MAE  = 1.1574
    RMSE = 1.6918

  alpha:
    R²   = 0.9978
    MAE  = 0.0061
    RMSE = 0.0074

  S_C2_C4 (%):
    R²   = 0.9992
    MAE  = 0.4295
    RMSE = 0.5248
